# Aula 05: Loop de Treinamento (Spiral Classification)

Neste lab, vamos aplicar tudo o que aprendemos: Tensores, Autograd, MLP e Otimizadores para resolver o problema da espiral (não linearmente separável).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Gerando o Dataset Espiral
Código clássico do curso CS231n.

In [ ]:
def create_spiral_dataset(N=100, K=3):
    X = np.zeros((N*K, 2))
    y = np.zeros(N*K, dtype='uint8')
    for j in range(K):
        ix = range(N*j,N*(j+1))
        r = np.linspace(0.0,1,N) # raio
        t = np.linspace(j*4,(j+1)*4,N) + np.random.randn(N)*0.2 # theta
        X[ix] = np.c_[r*np.sin(t), r*np.cos(t)]
        y[ix] = j
    return X, y

X_np, y_np = create_spiral_dataset(N=100, K=3)
plt.scatter(X_np[:,0], X_np[:,1], c=y_np, s=40, cmap=plt.cm.Spectral)
plt.show()

# Converter para Tensores
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.long)

## 2. O Modelo (MLP)
Crie uma `Sequential` com pelo menos 2 camadas ocultas. Experimente tamanhos (hidden_dim).

In [ ]:
hidden_dim = 100

model = nn.Sequential(
    nn.Linear(2, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim, 3) # 3 Classes
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

## 3. Training Loop

In [ ]:
epochs = 1000

for epoch in range(epochs):
    # 1. Forward
    logits = model(X)
    
    # 2. Loss
    loss = criterion(logits, y)
    
    # 3. Backward
    optimizer.zero_grad()
    loss.backward()
    
    # 4. Step
    optimizer.step()
    
    if epoch % 100 == 0:
        # Calcular acurácia
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        print(f"Epoch {epoch}: Loss {loss.item():.4f} | Acc {acc.item():.2f}")

## 4. Visualizando a Fronteira de Decisão
Vamos plotar o contorno de decisão para ver se o modelo aprendeu a espiral.

In [ ]:
def plot_decision_boundary(model, X, y):
    # Define limites
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    h = 0.02
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict para o grid inteiro
    grid_tensor = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    with torch.no_grad():
        Z = model(grid_tensor)
        Z = torch.argmax(Z, dim=1).numpy()
    
    Z = Z.reshape(xx.shape)
    plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, s=40, cmap=plt.cm.Spectral)
    plt.show()

plot_decision_boundary(model, X, y)